# 📓 02 — RFM 建模## 目标基于清洗后的交易数据，构建 RFM（Recency / Frequency / Monetary）三维用户价值模型。## RFM 含义| 维度 | 含义 | 业务价值 ||------|------|----------|| **R**ecency | 最近一次购买距今多久 | 客户活跃度，越小越好 || **F**requency | 购买频次 | 客户粘性 || **M**onetary | 消费总金额 | 客户价值 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append("../src")
from rfm import build_rfm_table, rfm_score, rfm_level
from data_loader import load_cleaned_retail  # pkl 找不到时会自动从原始数据重建
warnings = __import__("warnings")
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:.2f}".format)


## Step 1: 加载清洗后数据

In [ ]:
# pkl 51MB 超 GitHub 25MB 上限，仓库里不放。运行此 cell 会自动从原始数据重建
df = load_cleaned_retail()
print(f"数据规模: {df.shape}")
df.head()


## Step 2: 构建 RFM 基础表

In [ ]:
rfm = build_rfm_table(df)
print(f"客户数: {len(rfm)}")
rfm.describe()


## Step 3: 分布可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(rfm["Recency"], bins=50, ax=axes[0], color="steelblue", kde=True)
axes[0].set_title("Recency Distribution (days)")
axes[0].set_xlim(0, 400)

sns.histplot(rfm["Frequency"], bins=50, ax=axes[1], color="seagreen", kde=True)
axes[1].set_title("Frequency Distribution")
axes[1].set_xlim(0, 50)

sns.histplot(rfm["Monetary"], bins=50, ax=axes[2], color="indianred", kde=True)
axes[2].set_title("Monetary Distribution (GBP)")
axes[2].set_xlim(0, 10000)

plt.tight_layout()
plt.savefig("../images/rfm_distribution.png", dpi=120, bbox_inches="tight")
plt.show()


## Step 4: RFM 打分（1-5 分）

In [ ]:
rfm = rfm_score(rfm)
rfm["Customer_Level"] = rfm_level(rfm)

# R/F/M 平均得分
print("=== 各等级客户分布 ===")
print(rfm["Customer_Level"].value_counts())

print("\n=== RFM 得分示例 ===")
rfm.head(10)


## Step 5: 客户等级分布可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

level_order = ["Champion", "High", "Mid", "Low"]
counts = rfm["Customer_Level"].value_counts().reindex(level_order)

sns.barplot(x=counts.index, y=counts.values, ax=axes[0], palette="RdYlGn")
axes[0].set_title("Customer Level Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom")

# 占比饼图
axes[1].pie(counts.values, labels=counts.index, autopct="%1.1f%%",
            colors=["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"], startangle=90)
axes[1].set_title("Customer Level Share")

plt.tight_layout()
plt.savefig("../images/customer_level.png", dpi=120, bbox_inches="tight")
plt.show()


## Step 6: 导出 RFM 表

In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)
rfm.to_pickle("../data/processed/rfm_scored.pkl")
print(f"RFM 评分表已保存，{len(rfm)} 个客户")

# 关键统计
print("\n=== 客户价值画像 ===")
profile = rfm.groupby("Customer_Level")[["Recency", "Frequency", "Monetary"]].mean().round(2)
print(profile)


## ✅ 本节小结- 完成 RFM 三维建模，**R/F/M 三个维度各自分位数打分 1-5**- 综合分映射为 4 个客户等级：**Champion / High / Mid / Low**- RFM 评分表已导出至 `rfm_scored.pkl`，下一步进行聚类分析 👉 `03_clustering.ipynb`